# He - step 2: Cellucid export

Reads the intermediate written by `he_1_prepare_anndata.ipynb`, resolves
the gene names, and calls `cellucid.prepare` to export it. It
computes no embedding and no neighbour graph, so it is cheap to re-run after any
change to the export format, to the naming, or to the `prepare` signature - which
is the whole reason the build is split in two.

The published `exports-v3/he/` is built by
`python scripts/build_atlas.py --dataset he`, the pinned transcription of
this notebook and its step-1 sibling: it checks the SHA-256 of every input,
recomputes the graph and the layouts single-threaded and seeded, and refuses to
publish a generation that differs from `sources/he.json`. This notebook
is the annotated walk-through of the same recipe. It writes its export outside
every repository (`EXPORT_DIR` below), so running it never replaces the
published generation.

The publication sidecars of the published generation - `default.cellucid-session`
and `state-snapshots.json` - are **not** written here; they are captured against
the running application after a rebuild and published with
`scripts/publish_session_preset.mjs`, because the preset pins a digest of the
coordinates the build lays out.


In [ ]:
import os

# The gene export loop draws a tqdm progress bar on stderr, and committed
# notebook output is exactly what every reader of this file sees. The bar is
# disabled before cellucid imports tqdm; delete this line to watch progress
# when running interactively.
os.environ.setdefault("TQDM_DISABLE", "1")

import gzip
import json
import re
import resource
import sys
import time
from contextlib import contextmanager
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from cellucid import Embedding, prepare

TIMINGS: dict[str, float] = {}

# getrusage reports the high-water mark in bytes on macOS and in kibibytes
# everywhere else.
RSS_UNIT = 1 if sys.platform == "darwin" else 1024


def peak_rss_gb() -> float:
    """Peak resident set size of this kernel so far, in GB."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * RSS_UNIT / 1e9


@contextmanager
def stage(name: str):
    """Record the wall-clock cost of one export stage."""
    start = time.perf_counter()
    yield
    TIMINGS[name] = time.perf_counter() - start
    print(f"[{name}] {TIMINGS[name]:.1f} s  peak RSS {peak_rss_gb():.2f} GB")


print("cellucid", version("cellucid"))


In [ ]:
def find_repository_root(start: Path) -> Path:
    """Return the cellucid-datasets checkout that contains `start`."""
    for candidate in [start, *start.parents]:
        if (candidate / "exports-v3" / "datasets.json").is_file():
            return candidate
    raise RuntimeError(f"No cellucid-datasets checkout above {start}")


DATASET_ID = "he"
SOURCE_TITLE = "He"

REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
WORKSPACE_ROOT = REPOSITORY_ROOT.parent

SOURCE_DIR = Path(os.environ.get("CELLUCID_ANNDATA_DIR", WORKSPACE_ROOT / ".anndatas"))
INTERMEDIATE_DIR = Path(
    os.environ.get("CELLUCID_INTERMEDIATE_DIR", WORKSPACE_ROOT / ".anndatas-intermediate")
)
INTERMEDIATE_FILE = INTERMEDIATE_DIR / f"{DATASET_ID}_prepared.h5ad"

# Read-only upstream identity. The identifier frame is the one the integration
# pipeline produced; the symbol table is the only mapping this build may consult.
VARNAMES_FILE = SOURCE_DIR / "varnames" / f"dataset_complete_{SOURCE_TITLE}_varnames.pickle"
SYMBOL_TABLE_FILE = SOURCE_DIR / "ensembl_to_hgnc.pickle"

# Outside every repository: the published generation is built by
# scripts/build_atlas.py, and this walk-through never replaces it.
EXPORT_DIR = Path(
    os.environ.get("CELLUCID_NOTEBOOK_EXPORT_DIR", INTERMEDIATE_DIR / f"{DATASET_ID}_export")
).resolve()

if not INTERMEDIATE_FILE.is_file():
    raise FileNotFoundError(
        f"{INTERMEDIATE_FILE} is missing - run {DATASET_ID}_1_prepare_anndata.ipynb first."
    )


def show(path: Path) -> str:
    """Render a path relative to the workspace so committed output is portable."""
    try:
        return str(path.relative_to(WORKSPACE_ROOT))
    except ValueError:
        return str(path)


print("intermediate:", show(INTERMEDIATE_FILE),
      f"({INTERMEDIATE_FILE.stat().st_size / 1e6:.1f} MB)")
print("identifiers :", show(VARNAMES_FILE))
print("symbol table:", show(SYMBOL_TABLE_FILE))
print("export      :", show(EXPORT_DIR))


## Load the intermediate

Every parameter that shaped the embedding travels with the file, so the export
never has to restate them.


In [3]:
with stage("load_intermediate"):
    adata = ad.read_h5ad(INTERMEDIATE_FILE)

BUILD = dict(adata.uns["cellucid_build"])
for key, value in BUILD.items():
    print(f"{key:<20} {value}")
print()
print(adata)


[load_intermediate] 0.7 s  peak RSS 0.32 GB
anndata_version      0.12.19
created_at           2026-07-31T10:57:06Z
dataset_id           he
expression           raw integer counts, untransformed
feature_identity     positional_source_index; gene names resolved in step 2
label_normalization  none
latent_key           scvi
min_dist             0.5
n_neighbors          15
numpy_version        1.26.4
random_state         0
scanpy_version       1.11.5
source_bytes         298708152
source_file          he.h5ad

AnnData object with n_obs × n_vars = 71650 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', 'LVL3', 'LVL2', 'LVL1', 'LVL0'
    var: 'source_var_index'
    uns: 'cellucid_build'
    obsm: 'X_umap_1d', 'X_umap_2d', 'X_umap_3d', 'scvi'
    obsp: 'connectivities', 'distances'


## Gene names

**Every published gene is named by its symbol, and its upstream Ensembl accession
is published beside it as a second gene-name set**, `ensembl_gene_id`, which the
viewer's gene-names control switches to. Whatever this notebook writes is what the
gene search matches, what the legend prints, and what an analysis window has to be
given; `prepare` records the names it is handed and performs no lookup of its own.

The name has to be produced here, because the source carries none. Three files
meet in this cell:

1. `.anndatas/varnames/dataset_complete_He_varnames.pickle` - the
   identifier frame the integration pipeline produced, one entry per matrix
   column, in column order. It is a 0-column frame whose *index* is the
   identifier. Its length is asserted against `n_vars` before it is used: an
   identifier frame one row out of step mislabels every gene downstream of the
   offset while looking perfectly well-formed.
2. `.anndatas/ensembl_to_hgnc.pickle` - the only symbol mapping this build is
   allowed to consult. Nothing here reaches the network.
3. the intermediate's `var['source_var_index']`, which is the positional column
   number step 1 recorded, so the alignment above is checked rather than assumed.

### What counts as named

An identifier that is **already a symbol passes through unchanged and is never
looked up**. An `ENSG...` accession is resolved through the mapping, trying the
versioned key first and the unversioned base second.

**A gene is named only if the value it resolves to is not itself an Ensembl
accession.** The mapping contains 1,627 entries that map an accession to itself -
`'ENSG00000124835.2' -> 'ENSG00000124835.2'` - so a lookup can succeed and still
yield an accession. Keeping those would publish Ensembl identifiers as gene names,
which is exactly what this catalog no longer does, so they count as unnamed.
The test is on the resolved *value*: a mapping entry that resolves a symbol to
itself, `'CBWD3' -> 'CBWD3'`, is a real name and is kept.

### A gene with no name is not exported

This is a large, deliberate reduction of the published gene axis, and the summary
the cell prints is the record of it. An unnamed gene has no symbol a reader could
type into the gene search, and the catalog does not publish a gene under its
accession alone: the accession is offered only as the second name of a gene
that has a symbol. The dropped genes
stay in the source; they are absent from `exports-v3/`.


In [4]:
# An Ensembl gene accession, with or without its annotation version.
ENSEMBL_ACCESSION = re.compile(r"^ENSG\d")

identifier_frame = pd.read_pickle(VARNAMES_FILE)
assert list(identifier_frame.columns) == [], (
    f"identifier frame carries columns: {list(identifier_frame.columns)}"
)
assert len(identifier_frame) == adata.n_vars, (
    f"{VARNAMES_FILE.name} holds {len(identifier_frame):,} identifiers for "
    f"{adata.n_vars:,} matrix columns; a frame of a different length cannot be "
    f"aligned to the gene axis and would mislabel every gene past the offset"
)
assert identifier_frame.index.is_unique, "upstream identifiers are not unique"

# Step 1 recorded the positional column number, so the positional alignment of the
# identifier frame to the matrix is checked here rather than trusted.
assert adata.var["source_var_index"].tolist() == list(range(adata.n_vars))
assert adata.var.index.tolist() == [str(i) for i in range(adata.n_vars)]

upstream = [str(identifier) for identifier in identifier_frame.index]
symbol_by_accession = pd.read_pickle(SYMBOL_TABLE_FILE)
assert isinstance(symbol_by_accession, dict)


def resolve(identifier: str) -> str | None:
    """Resolve one upstream identifier to a gene symbol, or None."""
    if not ENSEMBL_ACCESSION.match(identifier):
        # Already a symbol. Looking it up could only replace a real name.
        return identifier
    if identifier in symbol_by_accession:
        return symbol_by_accession[identifier]
    return symbol_by_accession.get(identifier.split(".")[0])


resolved = [resolve(identifier) for identifier in upstream]
named = [
    value is not None and not ENSEMBL_ACCESSION.match(value)
    for value in resolved
]

# An unnamed gene keeps its upstream identifier here so that `var` still addresses
# every row uniquely; `gene_identifiers=` below is what keeps it out of the export.
gene_names = [
    value if keep else identifier
    for identifier, value, keep in zip(upstream, resolved, named)
]
adata.var["gene_name"] = gene_names
# The second gene-name set: the identifier exactly as the pipeline recorded it.
adata.var["ensembl_gene_id"] = upstream

NAMED_GENES = [name for name, keep in zip(gene_names, named) if keep]

passthrough = sum(
    1 for identifier, keep in zip(upstream, named)
    if keep and not ENSEMBL_ACCESSION.match(identifier)
)
translated = len(NAMED_GENES) - passthrough
still_accession = sum(
    1 for value, keep in zip(resolved, named) if not keep and value is not None
)
unmapped = sum(1 for value in resolved if value is None)

assert len(set(gene_names)) == len(gene_names), (
    "gene_name is not unique over var; two rows would be indistinguishable"
)
assert len(set(NAMED_GENES)) == len(NAMED_GENES), (
    "two genes resolved to the same name; one would hide the other in the viewer"
)
assert not [name for name in NAMED_GENES if ENSEMBL_ACCESSION.match(name)], (
    "a published gene name is still an Ensembl accession"
)

total = len(upstream)
rule = "=" * 66
print(rule)
print(f"GENE NAMING  {DATASET_ID}")
print(rule)
print(f"{total:>8,}  identifiers in the upstream var frame")
print(f"{passthrough:>8,}  already a symbol, passed through unchanged")
print(f"{translated:>8,}  Ensembl accessions resolved to a symbol")
print(f"{len(NAMED_GENES):>8,}  NAMED   -> published   "
      f"({len(NAMED_GENES) / total:.1%} of the source gene axis)")
print()
print(f"{still_accession:>8,}  resolved to an Ensembl accession, so still unnamed")
print(f"{unmapped:>8,}  absent from the symbol table")
print(f"{total - len(NAMED_GENES):>8,}  UNNAMED -> NOT published   "
      f"({(total - len(NAMED_GENES)) / total:.1%} of the source gene axis)")
print(rule)
print("first published names:", NAMED_GENES[:8])
print("dropped, first eight :",
      [identifier for identifier, keep in zip(upstream, named) if not keep][:8])


GENE NAMING  he
   8,192  identifiers in the upstream var frame
       6  already a symbol, passed through unchanged
   5,146  Ensembl accessions resolved to a symbol
   5,152  NAMED   -> published   (62.9% of the source gene axis)

     949  resolved to an Ensembl accession, so still unnamed
   2,091  absent from the symbol table
   3,040  UNNAMED -> NOT published   (37.1% of the source gene axis)
first published names: ['AC058791.1', 'AC062029.1', 'TNMD', 'FGR', 'CFH', 'GCLC', 'ENPP4', 'SEMA3F']
dropped, first eight : ['ENSG00000001626.18', 'ENSG00000004948.16', 'ENSG00000008438.5', 'ENSG00000011052.21', 'ENSG00000011083.9', 'ENSG00000011258.16', 'ENSG00000011590.14', 'ENSG00000012504.15']


## Constant-gene census

`prepare` publishes a quantized field as `[index, name, minValue, maxValue]`. A
gene carrying one value in every published cell is publishable: compact_v1 encodes
it as the constant case, `minValue == maxValue` with every code `0`, and the
viewer decodes that back to the exact value. A gene detected in none of the
published cells is ordinary data, not a reason to drop it.

The census below therefore selects nothing. `exportable` is the naming from the
previous cell and nothing else, so every named gene reaches the export. The census
runs so that the constant case is recorded where it is produced, rather than left
to be inferred downstream from a manifest entry whose two bounds happen to be
equal, and so that a gene detected in no cell is named in the committed output
instead of passing unremarked.

A continuous `obs` column can equally be flattened by a subset, and the writer
publishes it under the same encoding, so this cell reports each column's range
rather than requiring variation of it.


In [5]:
expression = adata.X.tocsc()
n_cells = adata.n_obs
nnz_per_gene = np.diff(expression.indptr)

minima = np.zeros(adata.n_vars, dtype=np.float32)
maxima = np.zeros(adata.n_vars, dtype=np.float32)
for index in range(adata.n_vars):
    values = expression.data[expression.indptr[index]:expression.indptr[index + 1]]
    values = values.astype(np.float32)
    if values.size:
        maxima[index] = values.max()
        minima[index] = values.min() if values.size == n_cells else np.float32(0.0)

constant = minima >= maxima
print(f"genes detected in no cell : {int((nnz_per_gene == 0).sum())}")
print(f"genes with no variation   : {int(constant.sum())}")
for index in np.flatnonzero(constant):
    print(f"  column {index:<5} upstream {upstream[index]:<20} "
          f"name {gene_names[index]:<20} "
          f"{'named' if named[index] else 'unnamed'}, detected in "
          f"{int(nnz_per_gene[index])} of {n_cells:,} cells")

exportable = np.asarray(named)
EXPORTED_GENES = [
    name for name, keep in zip(gene_names, exportable.tolist()) if keep
]
assert EXPORTED_GENES == NAMED_GENES, "naming is the only thing that selects genes"

continuous_obs = [
    key for key in adata.obs.columns
    if not hasattr(adata.obs[key], "cat")
]
for key in continuous_obs:
    values = adata.obs[key].to_numpy(dtype=np.float32)
    print(f"continuous obs {key!r}: [{values.min()}, {values.max()}]"
          + ("  constant" if values.min() == values.max() else ""))

constant_exported = int((exportable & constant).sum())
print(f"\n{len(EXPORTED_GENES):,} genes are exported at var_quantization=8: "
      f"every one of the {len(NAMED_GENES):,} named genes, "
      f"{constant_exported} of them as the constant case.")


genes detected in no cell : 0
genes with no variation   : 0
continuous obs 'age': [5.0, 22.0]
continuous obs 'n_genes': [10.0, 1356.0]

5,152 genes are exported at var_quantization=8: every one of the 5,152 named genes, 0 of them as the constant case.


## Export

Parameters are stated inline.

`var_gene_id_column='gene_name'` is the column built above, and
`gene_identifiers=EXPORTED_GENES` is the subset of it that is published. Every
other row of `var` is addressable in that column and simply not requested.
`var_gene_name_columns=['ensembl_gene_id']` publishes each published gene's
upstream accession as the second name set; it is metadata only, so no
expression is stored twice.

`embeddings` publishes the three UMAPs of the shared graph, the source file's
own 2D UMAP and the first two components of the unintegrated PCA, so the 2D
view offers three layouts; `embedding_defaults` opens it on the UMAP of the
shared graph.

`obs_categorical_dtype='uint8'` is exact rather than generous: the widest
categorical fields are `cell_type` and `LVL3` with 128 categories, inside the 255 that
8-bit codes address.

`latent_space` is the scVI representation the graph and the layouts
were built on, so the centroids and the outlier quantiles the viewer draws come
from the same space as the embedding.

`source_citation` is the attribution that travels with the accession: the
ArrayExpress study title and the accession itself. It names no author, year,
journal or DOI, because this build establishes none of them - the integration
output carries no bibliographic record and the provenance note names only the
accession - and a guessed reference would be worse than a narrow one.

In [ ]:
# `prepare` echoes every path it writes. Passing out_dir relative to this
# notebook keeps the committed log free of one machine's directory layout; it
# resolves to EXPORT_DIR against the working directory the kernel started in.
out_dir_arg = Path(os.path.relpath(EXPORT_DIR, Path.cwd()))
assert (Path.cwd() / out_dir_arg).resolve() == EXPORT_DIR

with stage("prepare"):
    prepare(
        # One shared kNN graph laid out in three dimensionalities (step 1),
        # the source file's own 2D UMAP, and the unintegrated PCA.
        embeddings={
            "X_umap_1d": Embedding(adata.obsm["X_umap_1d"], label="UMAP"),
            "X_umap_2d": Embedding(adata.obsm["X_umap_2d"], label="UMAP"),
            "X_umap_3d": Embedding(adata.obsm["X_umap_3d"], label="UMAP"),
            "X_umap": Embedding(
                adata.obsm["X_umap"], label="UMAP (source file)", source_key="X_umap"
            ),
            "X_pca": Embedding(
                adata.obsm["X_pca"], columns=[0, 1], label="PCA (unintegrated)",
                source_key="X_pca",
            ),
        },
        embedding_defaults={"2d": "X_umap_2d"},

        # scVI latent space: drives centroids and the outlier quantiles.
        latent_space=adata.obsm[BUILD["latent_key"]],
        obs=adata.obs,
        var=adata.var,
        gene_expression=adata.X,
        connectivities=adata.obsp["connectivities"],

        # The symbol is the default name, the accession the second name set,
        # and only genes with a symbol are published.
        var_gene_id_column="gene_name",
        var_gene_name_columns=["ensembl_gene_id"],
        gene_identifiers=EXPORTED_GENES,

        centroid_outlier_quantile=0.90,
        centroid_min_points=10,

        var_quantization=8,
        obs_continuous_quantization=8,
        obs_categorical_dtype="uint8",
        compression=6,

        out_dir=out_dir_arg,
        force=True,
        dataset_id=DATASET_ID,
        dataset_name="Developing human lung (He)",
        created_at=BUILD["created_at"],
        dataset_description="A human fetal lung cell atlas uncovers proximal-distal gradients of differentiation and key regulators of epithelial fates",
        source_name="E-MTAB-11278",
        source_url="https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-11278",
        source_citation="A human fetal lung cell atlas uncovers proximal-distal gradients of differentiation and key regulators of epithelial fates. ArrayExpress study E-MTAB-11278.",
    )


## Validate the export

Read back off disk, never from the call that wrote it.

The first cell checks what the viewer enforces at load time. The second checks the
payload identity rule the whole catalog is built on: **a filename is an integer
index or a fixed neutral name, and never dataset content.** Within one axis
directory the indices are exactly `0 ... N-1`, each used once, and for `obs` that
space is shared by the continuous and the categorical fields because both write
into `obs/`. Two fields sharing an index would silently overwrite one payload with
another and the viewer would draw one field's values under another field's name.


In [ ]:
identity = json.loads((EXPORT_DIR / "dataset_identity.json").read_text())
obs_manifest = json.loads((EXPORT_DIR / "obs_manifest.json").read_text())
var_manifest = json.loads((EXPORT_DIR / "var_manifest.json").read_text())
connectivity = json.loads((EXPORT_DIR / "connectivity_manifest.json").read_text())

stats = identity["stats"]
print(json.dumps({k: identity[k] for k in ("version", "id", "name", "created_at",
                                           "cellucid_data_version")}, indent=1))
print(json.dumps(stats, indent=1))

# 1. one manifest field per exported gene, in the order requested
assert len(var_manifest["fields"]) == stats["n_genes"] == len(EXPORTED_GENES)
assert [entry[1] for entry in var_manifest["fields"]] == EXPORTED_GENES
assert var_manifest["gene_name_sets"][0]["source"] == "gene_name"
assert [entry["source"] for entry in var_manifest["gene_name_sets"]] == [
    "gene_name", "ensembl_gene_id",
]
print(f"\nvar_manifest.fields == stats.n_genes == {len(EXPORTED_GENES):,} "
      f"of {adata.n_vars:,} source columns, in var order")

# 2. identity lists every continuous obs field before every categorical one
kinds = [field["kind"] for field in identity["obs_fields"]]
assert kinds == sorted(kinds, key=lambda kind: kind != "continuous")
assert kinds.count("continuous") == stats["n_continuous_fields"]
assert kinds.count("category") == stats["n_categorical_fields"]
print(f"obs_fields: {kinds.count('continuous')} continuous then "
      f"{kinds.count('category')} categorical")

# 3. every embedding declared is present on disk
items = identity["embeddings"]["items"]
for item in items.values():
    assert (EXPORT_DIR / item["file"]).is_file(), item["file"]
print("embeddings:", {key: item["dimension"] for key, item in items.items()},
      "defaults", identity["embeddings"]["defaults"])

# 4. connectivity edges are strictly ascending (source, destination) pairs
index_dtype = np.dtype(connectivity["index_dtype"]).newbyteorder("<")
sources = np.frombuffer(
    gzip.decompress((EXPORT_DIR / connectivity["sourcesPath"]).read_bytes()), dtype=index_dtype
)
destinations = np.frombuffer(
    gzip.decompress((EXPORT_DIR / connectivity["destinationsPath"]).read_bytes()), dtype=index_dtype
)
assert sources.size == destinations.size == connectivity["n_edges"]
assert np.all(sources < destinations), "every edge must run from the lower index"
order = np.lexsort((destinations, sources))
assert np.array_equal(order, np.arange(sources.size)), "edges must be lexicographically sorted"
print(f"connectivity: {connectivity['n_edges']:,} strictly ordered undirected edges, "
      f"max_neighbors {connectivity['max_neighbors']}")


In [ ]:
# The codes extension is the writer's own dtype-to-suffix table, not the dtype
# string the manifest entry carries.
CODES_EXTENSION = {"uint8": "u8", "uint16": "u16"}


def expand(pattern: str, index: int, ext: str = "") -> str:
    """One declared payload path, from the schema pattern the manifest publishes."""
    return pattern.replace("{index}", str(index)).replace("{ext}", ext)


# --- var: indices 0..N-1, names are names, payloads are what the manifest declares
var_schema = var_manifest["_varSchema"]
assert "{index}" in var_schema["pathPattern"], var_schema["pathPattern"]
assert "{key}" not in json.dumps(var_manifest)
var_indices = [entry[0] for entry in var_manifest["fields"]]
assert var_indices == list(range(len(var_manifest["fields"]))), "var indices are not 0..N-1"
assert all(len(entry) == 4 for entry in var_manifest["fields"]), "quantized var entry is [index, name, min, max]"
var_names = [entry[1] for entry in var_manifest["fields"]]
assert len(set(var_names)) == len(var_names), "a published gene name is repeated"
assert not [name for name in var_names if ENSEMBL_ACCESSION.match(name)], (
    "a published gene name is an Ensembl accession"
)
declared_var = {
    expand(var_schema["pathPattern"], entry[0], var_schema["ext"])
    for entry in var_manifest["fields"]
}
on_disk_var = {
    path.relative_to(EXPORT_DIR).as_posix() for path in (EXPORT_DIR / "var").rglob("*")
    if path.is_file()
}
assert declared_var == on_disk_var, "var/ is not exactly what var_manifest.json declares"
print(f"var: indices 0..{len(var_indices) - 1}, {len(on_disk_var):,} payloads, "
      f"declared set == on-disk set")

# --- obs: one index space shared by both field arrays
obs_schemas = obs_manifest["_obsSchemas"]
assert '{key}' not in json.dumps(obs_manifest)
continuous_entries = obs_manifest["_continuousFields"]
categorical_entries = obs_manifest["_categoricalFields"]
obs_indices = [entry[0] for entry in continuous_entries + categorical_entries]
assert sorted(obs_indices) == list(range(len(obs_indices))), (
    f"obs indices are not 0..N-1 exactly once across both field arrays: {sorted(obs_indices)}"
)
assert all(len(entry) == 4 for entry in continuous_entries)
assert all(len(entry) == 8 for entry in categorical_entries)
declared_obs = {
    expand(obs_schemas["continuous"]["pathPattern"], entry[0], obs_schemas["continuous"]["ext"])
    for entry in continuous_entries
}
for entry in categorical_entries:
    schema = obs_schemas["categorical"]
    declared_obs.add(expand(schema["codesPathPattern"], entry[0], CODES_EXTENSION[entry[3]]))
    declared_obs.add(expand(schema["outlierPathPattern"], entry[0]))
on_disk_obs = {
    path.relative_to(EXPORT_DIR).as_posix() for path in (EXPORT_DIR / "obs").rglob("*")
    if path.is_file()
}
assert declared_obs == on_disk_obs, "obs/ is not exactly what obs_manifest.json declares"
print(f"obs: {len(continuous_entries)} continuous + {len(categorical_entries)} categorical "
      f"share the indices 0..{len(obs_indices) - 1}, {len(on_disk_obs)} payloads, "
      f"declared set == on-disk set")

# --- connectivity and the embeddings: fixed neutral names, no dataset content
NEUTRAL = {
    "connectivity/edges.src.bin.gz",
    "connectivity/edges.dst.bin.gz",
    "connectivity/edges.weights.f64.bin.gz",
}
assert {
    path.relative_to(EXPORT_DIR).as_posix()
    for path in (EXPORT_DIR / "connectivity").rglob("*") if path.is_file()
} == NEUTRAL
ROOT_FILES = {
    "dataset_identity.json", "obs_manifest.json", "var_manifest.json",
    "connectivity_manifest.json",
}
assert {
    path.name for path in EXPORT_DIR.iterdir() if path.is_file()
} == ROOT_FILES
# Every coordinate payload lives under embeddings/, named by its item index.
assert {
    path.relative_to(EXPORT_DIR).as_posix()
    for path in (EXPORT_DIR / "embeddings").rglob("*") if path.is_file()
} == {item["file"] for item in identity["embeddings"]["items"].values()}

# --- no filename anywhere in the tree spells a gene name or an obs key
content = {*var_names, *[entry[1] for entry in continuous_entries + categorical_entries],
           *upstream}
offenders = [
    path.relative_to(EXPORT_DIR).as_posix()
    for path in EXPORT_DIR.rglob("*") if path.is_file()
    if any(token in path.name for token in content)
]
assert not offenders, f"a filename carries dataset content: {offenders[:5]}"
print(f"connectivity and root files are the fixed neutral set; no path in the tree "
      f"spells any of the {len(content):,} identifiers this dataset carries")


In [ ]:
# Round-trip one gene: the exported codes must be exactly what the documented
# quantization produces from the intermediate's own column, and the file that
# holds them is reached by index while the manifest beside it says which gene
# that index is. Every constant gene is round-tripped as well, because the
# constant case is the one the general arithmetic here cannot express.
def round_trip(entry: list) -> str:
    """Reproduce one published gene payload from the intermediate's own column."""
    payload_index, name, low, high = entry
    codes = np.frombuffer(
        gzip.decompress(
            (EXPORT_DIR / expand(var_schema["pathPattern"], payload_index, var_schema["ext"]))
            .read_bytes()
        ),
        dtype=np.uint8,
    )
    source_column = int(adata.var.index[adata.var["gene_name"] == name][0])
    column = expression.getcol(source_column).toarray().ravel().astype(np.float32)
    if low == high:
        # compact_v1's constant case: the two bounds carry the value itself and
        # every code is 0, so there is no scale here and nothing to divide by.
        assert np.array_equal(column, np.full(column.size, low, dtype=np.float32))
        expected = np.zeros(column.size, dtype=np.uint8)
    else:
        expected = np.clip(
            (column.astype(np.float64) - low) * (254.0 / (high - low)) + 1e-8, 0, 254
        ).astype(np.uint8)
    assert np.array_equal(codes, expected), f"{name} does not round-trip"
    return (f"gene {name!r} is payload index {payload_index} (source column "
            f"{source_column}, upstream {upstream[source_column]!r}) and round-trips "
            f"exactly over {codes.size:,} cells, bounds {low} - {high}"
            + ("  [constant case]" if low == high else ""))


print(round_trip(var_manifest["fields"][0]))
for entry in var_manifest["fields"]:
    if entry[2] == entry[3]:
        print(round_trip(entry))


In [ ]:
files = sorted(path for path in EXPORT_DIR.rglob("*") if path.is_file())
total_bytes = sum(path.stat().st_size for path in files)
print(f"{len(files):,} files, {total_bytes / 1e6:.1f} MB in {show(EXPORT_DIR)}")
for path in sorted(EXPORT_DIR.iterdir()):
    if path.is_dir():
        inner = [p for p in path.rglob('*') if p.is_file()]
        size = sum(p.stat().st_size for p in inner)
        print(f"  {path.name + '/':<26} {len(inner):>6,} files  {size / 1e6:8.2f} MB")
    else:
        print(f"  {path.name:<26} {'':>6}         {path.stat().st_size / 1e6:8.2f} MB")

print()
total = sum(TIMINGS.values())
for name, seconds in TIMINGS.items():
    print(f"{name:<22} {seconds:8.1f} s")
print(f"{'TOTAL':<22} {total:8.1f} s")
print(f"{'peak RSS':<22} {peak_rss_gb():8.2f} GB")


The export under `EXPORT_DIR` is complete and internally consistent.

It is the walk-through's own artifact. The published `exports-v3/he/` is
built by `scripts/build_atlas.py --dataset he`, and its starting view
is captured against the running application and published with
`scripts/publish_session_preset.mjs`, which also repins
`exports-v3/datasets.json`.